In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load dataset
df = pd.read_csv("Preprocessed_Dataset_For_Prediction.csv")
df = pd.get_dummies(df, drop_first=True)

In [3]:
df.columns

Index(['Unnamed: 0', 'mrr_amount', 'seats', 'upgrade_flag', 'downgrade_flag',
       'churn_flag', 'auto_renew_flag', 'usage_count', 'error_count',
       'error_rate', 'resolution_time_hours', 'satisfaction_score',
       'escalation_flag', 'no_feedback', 'no_ticket'],
      dtype='object')

In [4]:
df.head(2)

,Unnamed: 0,mrr_amount,seats,upgrade_flag,downgrade_flag,churn_flag,auto_renew_flag,usage_count,error_count,error_rate,resolution_time_hours,satisfaction_score,escalation_flag,no_feedback,no_ticket
0,0,3350.6,34.200000,3,0,False,0.900000,514,27,0.052427,31.75,4.000000,0.0,0,0
1,1,1569.0,15.333333,0,1,False,0.888889,602,31,0.051410,33.00,3.964163,0.0,1,0


In [5]:
X = df.drop("churn_flag", axis=1)
y = df["churn_flag"]

In [6]:
print(f"Original dataset shape: {X.shape}")

Original dataset shape: (500, 14)


In [7]:
# Remove low variance features
vt = VarianceThreshold(threshold=0.01)
X_reduced = vt.fit_transform(X)
X_columns_reduced = X.columns[vt.get_support()]
X = pd.DataFrame(X_reduced, columns=X_columns_reduced)
print(f"After variance threshold: {X.shape}")

After variance threshold: (500, 12)


In [8]:
print(X_columns_reduced)

Index(['Unnamed: 0', 'mrr_amount', 'seats', 'upgrade_flag', 'downgrade_flag',
       'auto_renew_flag', 'usage_count', 'error_count',
       'resolution_time_hours', 'satisfaction_score', 'escalation_flag',
       'no_feedback'],
      dtype='object')


In [9]:
# Step 3 (Optional): Sample dataset if large
if X.shape[0] > 10000:
    sample_idx = np.random.choice(X.index, size=5000, replace=False)
    X_sample = X.loc[sample_idx]
    y_sample = y.loc[sample_idx]
else:
    X_sample = X
    y_sample = y

In [30]:
# Step 4: Model switch for feature selection
use_model = "random_forest"  # options: "logistic", "decision_tree", "random_forest"

if use_model == "logistic":
    print("Using Logistic Regression for feature selection...")
    estimator = LogisticRegression(max_iter=1000)
elif use_model == "decision_tree":
    print("Using Decision Tree for feature selection...")
    estimator = DecisionTreeClassifier(random_state=0)
elif use_model == "random_forest":
    print("Using Random Forest for feature selection...")
    estimator = RandomForestClassifier(n_estimators=100, random_state=0)
else:
    raise ValueError("Invalid model selection for feature selection")


# Perform forward feature selection
selector = SequentialFeatureSelector(
    estimator,
    n_features_to_select=6,
    direction="forward",
    cv=3,
    n_jobs=-1
)
selector.fit(X_sample, y_sample)

selected_features = X.columns[selector.get_support()]
X_selected = X[selected_features]
print("Selected features:", list(selected_features))

Using Random Forest for feature selection...
Selected features: ['upgrade_flag', 'downgrade_flag', 'usage_count', 'error_count', 'escalation_flag', 'no_feedback']


In [31]:
print("Random Forest Selected features: ['Trial status', 'upgrade_flag', 'downgrade_flag', 'auto_renew_flag', 'industry_DevTools', 'industry_HealthTech']")

Random Forest Selected features: ['Trial status', 'upgrade_flag', 'downgrade_flag', 'auto_renew_flag', 'industry_DevTools', 'industry_HealthTech']


In [32]:
# Step 5: Train-test split and scale
def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_and_scale(X_selected, y)

In [33]:
# Step 6: Classifier wrapper
def evaluate_model(classifier, name):
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {acc:.4f}")
    return {
        "Model": name,
        "Accuracy": acc,
        "Report": classification_report(y_test, y_pred, output_dict=False),
        "Confusion Matrix": confusion_matrix(y_test, y_pred)
    }

In [34]:
# Step 7: Run all models
results = []
results.append(evaluate_model(LogisticRegression(max_iter=1000), "Logistic Regression"))
results.append(evaluate_model(SVC(kernel='linear'), "SVM (Linear)"))
results.append(evaluate_model(SVC(kernel='rbf'), "SVM (RBF)"))
results.append(evaluate_model(GaussianNB(), "Naive Bayes"))
results.append(evaluate_model(KNeighborsClassifier(n_neighbors=5), "KNN"))
results.append(evaluate_model(DecisionTreeClassifier(criterion='entropy'), "Decision Tree"))
results.append(evaluate_model(RandomForestClassifier(n_estimators=100, criterion='entropy'), "Random Forest"))

Logistic Regression Accuracy: 0.6080
SVM (Linear) Accuracy: 0.5680
SVM (RBF) Accuracy: 0.5680
Naive Bayes Accuracy: 0.6080
KNN Accuracy: 0.5760
Decision Tree Accuracy: 0.5600
Random Forest Accuracy: 0.6480


In [29]:
# Step 8: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"})

print("\nModel Accuracy Comparison #4:")
print(summary_df)


Model Accuracy Comparison #4:
                     Accuracy
Logistic Regression     0.560
SVM (Linear)            0.568
SVM (RBF)               0.544
Naive Bayes             0.568
KNN                     0.560
Decision Tree           0.656
Random Forest           0.632


In [23]:
# Step 8: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"})

print("\nModel Accuracy Comparison #5:")
print(summary_df)


Model Accuracy Comparison #5:
                     Accuracy
Logistic Regression     0.576
SVM (Linear)            0.568
SVM (RBF)               0.552
Naive Bayes             0.568
KNN                     0.560
Decision Tree           0.632
Random Forest           0.616


In [35]:
# Step 8: Summarize accuracy
summary_df = pd.DataFrame({
    result["Model"]: [result["Accuracy"]] for result in results
}).T.rename(columns={0: "Accuracy"})

print("\nModel Accuracy Comparison #6:")
print(summary_df)


Model Accuracy Comparison #6:
                     Accuracy
Logistic Regression     0.608
SVM (Linear)            0.568
SVM (RBF)               0.568
Naive Bayes             0.608
KNN                     0.576
Decision Tree           0.560
Random Forest           0.648
